In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import eigs, eigsh
from scipy import linalg
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import accuracy_score
from scipy.optimize import linear_sum_assignment

In [3]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh

def bethe_hessian_matrix(A, r):
    """
    Compute the Bethe Hessian matrix for a given adjacency matrix A and regularizer r.
    
    Args:
    A (scipy sparse matrix): adjacency matrix
    r (float): regularization parameter
    
    Returns:
    H (scipy sparse matrix): Bethe Hessian matrix
    """
    degrees = np.sum(A, axis=1)
    n = A.shape[0]
    
    # Create a diagonal degree matrix
    D = diags(degrees)
    
    # Bethe Hessian matrix: H = (r^2 - 1) * I - r * A + D
    H = (r**2 - 1) * np.identity(n) - r * A + D
    
    return H

def estimate_number_of_communities(A):
    """
    Estimate the number of communities in a network using the Bethe Hessian matrix.
    
    Args:
    A (scipy sparse matrix): adjacency matrix
    
    Returns:
    int: estimated number of communities
    """
    n = A.shape[0]
    degrees = np.sum(A, axis=1)
    # Compute the average degree (used to set the regularizer r)
    avg_degree = np.mean(A.sum(axis=1))
    
    # Set the regularization parameter r
    r = np.sqrt(avg_degree)
    # r = np.sqrt(np.sum(degrees**2) / np.sum(degrees) - 1)

    # Compute the Bethe Hessian matrix
    H = bethe_hessian_matrix(A, r)
    
    # Compute the eigenvalues of the Bethe Hessian matrix
    eigvals = linalg.eigvals(H)
    eigvals = np.sort(eigvals)[::-1]

    # plot the eigenvalues
    plt.plot(eigvals.real, eigvals.imag, 'o')
    plt.xlabel('Re')
    plt.ylabel('Im')
    plt.title('Eigenvalues of the Bethe Hessian matrix')
    plt.show()
    
    # Count the number of communities based on the eigenvalues
    t = 5
    for i in range(0, len(eigvals)-1):
        if eigvals[i] >= t * eigvals[i+1]:
            return n - i
    
    return 0

In [4]:
# cites_file = "Datasets/citeseer-doc-classification/citeseer.cites"
# content_file = "Datasets/citeseer-doc-classification/citeseer.content"
# A, features, actual_labels = parse_dataset(cites_file, content_file)
# k1 = estimate_number_of_communities(A)
# print(f"Estimated number of communities using Bethes Hessian: {k1}")
# # k2 = estimate_num_communities(A)
# # print(f"Estimated number of communities using Non-Backtracking: {k2}")

In [28]:
# Non-backtracking matrix-based method to estimate number of communities
def estimate_num_communities(A, initial_k=4, step=1, max_k=10):
    # Step 1: Calculate the degrees of the nodes
    degrees = np.sum(A, axis=1)
    diag_degrees = diags(degrees)
    
    # Step 2: Construct the non-backtracking matrix
    non_backtracking_matrix = np.block([
        [np.zeros_like(A), diag_degrees - np.identity(len(A))],
        [-np.identity(len(A)), A]
    ])
    
    # Step 3: Compute the norm of the non-backtracking matrix
    # radius = np.sqrt(np.linalg.norm(non_backtracking_matrix, ord=2))
    radius = np.sqrt(np.sum(degrees**2) / np.sum(degrees) - 1) # Approximation of the spectral radius
    print("Radius of the non-backtracking matrix:", radius)
    # radius = np.sqrt(np.sum(np.abs(non_backtracking_matrix.data)**2))

    k = initial_k
    communities_count = 0
    all_real_found = False

    eigvals = linalg.eigvals(non_backtracking_matrix)
    # eigvals = np.sort(eigvals)[::-1]
    # print the eigenvalues in a file
    np.savetxt('eigenvalues.txt', eigvals)
    # plot the eigenvalues
    plt.plot(eigvals.real, eigvals.imag, 'o')
    plt.xlabel('Re')
    plt.ylabel('Im')
    plt.title('Eigenvalues of the non-backtracking matrix')
    plt.show()

    # count number of real eigenvalues that are greater than or equal to the radius
    communities_count = len([eig for eig in eigvals if np.isclose(eig.imag, 0) and abs(eig.real) >= radius])
    
    # while not all_real_found and k <= max_k:
    # # while k <= max_k:
    #     # Step 4: Compute the eigenvalues of the non-backtracking matrix
    #     # sort the eigenvalues in descending order
    #     # get the top k eigenvalues
    #     k_eigvals = eigvals[:k]
    #     print(k_eigvals)
    #     # print(radius)
        
    #     # Step 5: Filter out eigenvalues inside the circle with radius equal to the norm of the matrix
    #     eigen_distances = np.abs(k_eigvals)  # Magnitude of each eigenvalue
    #     outside_circle = eigen_distances > radius
        
    #     # Count the number of eigenvalues outside the circle
    #     communities_count = np.sum(outside_circle)
    #     print(f"Number of eigenvalues outside the circle: {communities_count}")
    #     if communities_count > 0:
    #        all_real_found = True  # Stop if we have enough eigenvalues to count
        
    #     else:
    #         # Increase k and try again
    #         k += step

    return communities_count

# Function to initialize P randomly
def initialize_P(n, k):
    P = np.abs(np.random.rand(n, k))
    for i in range(n):
        if np.sum(P[i]) == 0:
            P[i] = np.ones(k) / k
        else:
            P[i] /= np.sum(P[i])
    return P

# Symmetric NMF update rule
def update_P(Lz, P, beta=0.5):
    numerator = np.dot(Lz, P)
    denominator = np.dot(np.dot(P, P.T), P)
    denominator[denominator == 0] = 1e-10  # Avoid division by zero
    P = P * ((1 - beta) + beta * numerator / denominator)
    P_updated = np.maximum(np.dot(np.dot(Lz, P), np.linalg.inv(np.dot(P.T, P))), 0) # Non-negative constraint
    return P_updated

def calculate_ipr(vector):
    """
    Calculate the Inverse Participation Ratio (IPR) for a given vector.
    IPR is the sum of the fourth power of the elements in the vector.
    """
    return np.sum(vector**4)

def update_regularization_matrix(Z, feature_matrix, threshold=0.1):
    """
    Update the diagonal elements of matrix Z based on the IPR of the feature vectors.
    A penalty is applied if the IPR exceeds a certain threshold (indicating high localization).
    
    Parameters:
    - Z: Initial regularization matrix (diagonal, 4x4 in this case for 4 nodes).
    - feature_matrix: The matrix where each row is a feature vector (10x4).
    - threshold: A value to decide when to apply a penalty based on IPR.
    """
    for i, vector in enumerate(feature_matrix):  # Iterate over the columns (nodes' feature vectors)
        ipr_value = calculate_ipr(vector)
        if ipr_value > threshold:
            # print(f"Node {i} has high IPR ({ipr_value}), applying penalty.")
            # Apply a penalty by modifying the diagonal element of Z
            Z[i, i] -= ipr_value  # Example penalty: increase the diagonal element by 1
    return Z

def learn_regularization_matrix(A, Z, num_iterations=10, penalty_factor=0.1):
    for _ in range(num_iterations):
        # Step 3.1: Compute the Z-Laplacian matrix
        Lz = A + Z
        
        # Step 3.2: Compute eigenvectors of the Z-Laplacian
        eigenvalues, eigenvectors = np.linalg.eig(Lz)
        
        # Step 3.3: Compute IPR (Inverse Participation Ratio) for each eigenvector
        IPR = np.sum(eigenvectors ** 4, axis=0)  # Higher IPR -> More localized
        
        # Step 3.4: Sort by IPR to find the most localized eigenvectors
        localized_indices = np.argsort(-IPR)  # Sort in descending order of localization
        
        # Step 3.5: Penalize the most localized eigenvectors by updating Z
        for idx in localized_indices[:5]:  # Penalize top 5 localized eigenvectors
            Z[idx, idx] -= penalty_factor
    
    return Z

# CDNMF algorithm
def CDNMF(A, features, k, num_iterations=100, beta=0.5):
    n = A.shape[0]
    
    # Step 1: Estimate the number of communities (k) using spectral analysis
    # k = estimate_num_communities(A)
    # k = 8  # For simplicity, we will use a fixed number of communities
    print(f"Estimated number of communities: {k}")
    
    # Step 2: Initialize P matrix randomly
    P = initialize_P(n, k)
    
    # Step 3: Initialize regularization matrix Z
    Z = np.eye(n)
    Z = update_regularization_matrix(Z, features, threshold=0.001)
    # Z = learn_regularization_matrix(A, Z, num_iterations=10, penalty_factor=0.1)

    # Step 4: Compute regularized Laplacian Lz
    Lz = A + Z
    
    # Step 5: Iteratively update P using the update rule
    prev_obj_value = -np.inf
    threshold_value = np.linalg.norm(A, 'fro') * 1e-5
    for iteration in range(num_iterations):
        P = update_P(Lz, P, beta)

        # Compute the objective function value
        obj_value = np.linalg.norm(A - np.dot(P, P.T), 'fro')
        if iteration % 10 == 0:
            print(f"Objective function value after {iteration+1} iterations: {obj_value}")

        # If the change in objective value is below a certain threshold, stop the iterations
        if abs(prev_obj_value - obj_value) < threshold_value:
            print(f"Converged after {iteration+1} iterations.")
            break
        prev_obj_value = obj_value

    # Normalize P so that each row sums to 1
    # if some rows are all zeros, set them to uniform distribution
    for i in range(n):
        if np.sum(P[i]) == 0:
            P[i] = np.ones(k) / k
        else:
            P[i] /= np.sum(P[i])
    
    # Assign nodes to communities based on maximum membership value
    communities = np.argmax(P, axis=1)
    
    return communities, P


In [6]:
# Function to parse .cites and .content files and build the adjacency matrix
def parse_dataset(cites_file, content_file):
    # Parse the .content file
    features = []
    node_id_to_index = {}
    class_labels = {}
    actual_labels = []
    num_nodes = 0
    with open(content_file, "r") as file:
        for line in file:
            parts = line.strip().split("\t")
            node_id = parts[0]
            node_id_to_index[node_id] = num_nodes

            feature_vector = np.array([float(x) for x in parts[1:-1]])
            feature_vector = feature_vector / np.linalg.norm(feature_vector)
            features.append(feature_vector)

            class_label = parts[-1]
            if class_label not in class_labels:
                class_labels[class_label] = len(class_labels)
            actual_labels.append(class_labels[class_label])
            
            num_nodes += 1

    # make number of features consistent
    max_features = max([len(feature) for feature in features])
    for i in range(len(features)):
        features[i] = np.pad(features[i], (0, max_features - len(features[i])), 'constant')
    
    features = np.array(features)

    # Parse the .cites file
    edges = []
    with open(cites_file, "r") as file:
        for line in file:
            parts = line.strip().split("\t")
            edges.append((parts[0], parts[1]))
    
    # Build the adjacency matrix
    A = np.zeros((num_nodes, num_nodes))
    for edge in edges:
        if edge[0] not in node_id_to_index or edge[1] not in node_id_to_index:
            # print(f"Node not found: {edge[0]} or {edge[1]}")
            continue
        x = node_id_to_index[edge[0]]
        y = node_id_to_index[edge[1]]
        A[x, y] = 1
        A[y, x] = 1
        # print(f"Adding edge between {x} and {y}")

    # Remove rows and columns with all zeros
    degrees = np.sum(A, axis=1)
    zero_rows = np.where(degrees == 0)[0]
    A = np.delete(A, zero_rows, axis=0)
    zero_cols = np.where(np.sum(A, axis=0) == 0)[0]
    A = np.delete(A, zero_cols, axis=1)
    
    print(f"Number of nodes: {A.shape[0]}")
    print(f"Number of edges: {np.sum(A) // 2}")
    print(f"Feature dimension: {len(features[0])}")
    print(f"Number of unique class labels: {len(class_labels)}")

    # Count number of isolated nodes
    isolated_nodes = np.where(np.sum(A, axis=1) == 0)[0]
    print(f"Number of isolated nodes: {len(isolated_nodes)}")

    # # Convert directed graph to undirected graph by duplicating the nodes
    # A = np.zeros((2*num_nodes, 2*num_nodes))
    # for edge in edges:
    #     if edge[0] not in node_id_to_index or edge[1] not in node_id_to_index:
    #         continue
    #     x = node_id_to_index[edge[0]]
    #     y = node_id_to_index[edge[1]]
    #     A[x, y] = 1
    #     A[y, x] = 1
    #     A[x, x+num_nodes] = 1
    #     A[x+num_nodes, x] = 1
    #     A[y, y+num_nodes] = 1
    #     A[y+num_nodes, y] = 1
    #     A[x+num_nodes, y+num_nodes] = 1
    #     A[y+num_nodes, x+num_nodes] = 1
    #     A[x, y+num_nodes] = 1
    #     A[y+num_nodes, x] = 1

    # features = np.concatenate((features, features), axis=0)
    # actual_labels = actual_labels + actual_labels
    
    return A, features, actual_labels

In [7]:
def calculate_modularity(A, communities):
    """
    Calculate modularity of a graph with adjacency matrix A and community assignments.
    
    A: numpy array (adjacency matrix)
    communities: list or array of community assignments for each node
    """
    num_nodes = A.shape[0]
    m = np.sum(A) // 2  # Total number of edges in the graph
    modularity = 0.0

    # Calculate degree of each node
    degrees = np.sum(A, axis=1)
    
    for i in range(num_nodes):
        for j in range(num_nodes):
            if communities[i] == communities[j]:  # Only consider nodes in the same community
                modularity += float(A[i, j]) - float(degrees[i] * degrees[j]) / float(2 * m)
    
    return modularity / (2 * m)

In [8]:
from math import comb

# Function to compute the Adjusted Rand Index (ARI)
def adjusted_rand_index(labels_true, labels_pred):
    n = len(labels_true)
    
    # Create contingency table (confusion matrix)
    contingency = {}
    for i in range(n):
        true_label = labels_true[i]
        pred_label = labels_pred[i]
        if (true_label, pred_label) in contingency:
            contingency[(true_label, pred_label)] += 1
        else:
            contingency[(true_label, pred_label)] = 1
    
    # Marginal sums for the true and predicted clusters
    sum_true = {}
    sum_pred = {}
    for (true_label, pred_label), count in contingency.items():
        if true_label in sum_true:
            sum_true[true_label] += count
        else:
            sum_true[true_label] = count
        
        if pred_label in sum_pred:
            sum_pred[pred_label] += count
        else:
            sum_pred[pred_label] = count
    
    # Calculate index sums
    sum_comb_cij = sum(comb(count, 2) for count in contingency.values())
    sum_comb_ai = sum(comb(count, 2) for count in sum_true.values())
    sum_comb_bj = sum(comb(count, 2) for count in sum_pred.values())
    
    # Calculate the ARI
    index = sum_comb_cij
    expected_index = sum_comb_ai * sum_comb_bj / comb(n, 2)
    max_index = (sum_comb_ai + sum_comb_bj) / 2
    
    ari = (index - expected_index) / (max_index - expected_index)
    return ari

# # Example usage
# labels_true = [0, 0, 1, 1, 2, 2]
# labels_pred = [0, 0, 1, 2, 2, 2]

# ari = adjusted_rand_index(labels_true, labels_pred)
# print(f'Adjusted Rand Index: {ari}')


In [32]:
def calculate_accuracy(C1, C2):
    return accuracy_score(C1, C2)

def best_accuracy(C1, C2):
    # Get unique labels from both sets
    labels_C1 = np.unique(C1)
    labels_C2 = np.unique(C2)
    
    # Build a cost matrix for the assignment
    cost_matrix = np.zeros((len(labels_C1), len(labels_C2)))
    
    for i, label_true in enumerate(labels_C1):
        for j, label_pred in enumerate(labels_C2):
            # Negative accuracy (we use a cost matrix so lower is better)
            cost_matrix[i, j] = -np.sum((np.array(C1) == label_true) & (np.array(C2) == label_pred))
    
    # Find the best alignment using the Hungarian algorithm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # Remap the predicted labels to best match true labels
    C2_mapped = np.zeros_like(C2)
    for i, j in zip(row_ind, col_ind):
        C2_mapped[np.array(C2) == labels_C2[j]] = labels_C1[i]
    
    # Now calculate the accuracy with the best-matched labels
    return accuracy_score(C1, C2_mapped)

In [ ]:
# Example usage
if __name__ == "__main__":
    # Load the dataset
    # cites_file = "Datasets/cora/cora.cites"
    # content_file = "Datasets/cora/cora.content"
    cites_file = "Datasets/citeseer-doc-classification/citeseer.cites"
    content_file = "Datasets/citeseer-doc-classification/citeseer.content"
    A, features, actual_labels = parse_dataset(cites_file, content_file)
    unique_labels = len(set(actual_labels))
    print(f"Actual labels: {actual_labels}")

    # Run the CDNMF algorithm
    communities, P = CDNMF(A, features, k=6, num_iterations=100, beta=0.5) # Best values of num_iterations and beta are 100 and 0.5 respectively
    
    # Output community assignment
    print("Community assignment:", communities)
    # print("Community membership matrix P:\n", P)

    # Calculate modularity
    calc_modularity = calculate_modularity(A, communities)
    orig_modularity = calculate_modularity(A, actual_labels)
    print(f"Original modularity: {orig_modularity}")
    print(f"Calculated modularity: {calc_modularity}")

    # Calculate the Adjusted Rand Index
    ari = adjusted_rand_index(actual_labels, communities)
    print(f'Adjusted Rand Index: {ari}')

    # print the results in a single file
    with open('results.txt', 'a') as f:
        f.write(f"Dataset: {cites_file}\n")
        f.write(f"Community assignment: {communities}\n")
        f.write(f"Original modularity: {orig_modularity}\n")
        f.write(f"Calculated modularity: {calc_modularity}\n")
        f.write(f'Adjusted Rand Index: {ari}\n')
        f.write("-"*50 + "\n")

In [35]:
import networkx as nx
import matplotlib.pyplot as plt

# Parameters for the LFR benchmark
n = 3000            # Number of nodes
tau2 = 1.5         # Power-law exponent for the community size distribution
mu = 0.15           # Mixing parameter (fraction of edges that connect outside communities)
avg_degree = 4    # Minimum degree of nodes
max_degree = 15    # Maximum degree of nodes
min_community = 50 # Minimum size of communities
max_community = 100 # Maximum size of communities

# Derive min_degree based on avg_degree. (Note: min_degree must be <= avg_degree)
min_degree = max(1, avg_degree - 5)

# Estimate tau1 (degree distribution exponent)
tau1 = 3

# Generate LFR benchmark graph
G = nx.generators.community.LFR_benchmark_graph(
    n, tau1, tau2, mu, min_degree=min_degree, max_degree=max_degree, 
    min_community=min_community, max_community=max_community, seed=42
)

# Remove isolated nodes, if any
G.remove_nodes_from(list(nx.isolates(G)))

# Get the adjacency matrix
adj_matrix = nx.to_numpy_array(G)

# Initialize the 1D community assignment array (size: n)
community_assignment = np.zeros(n, dtype=int)

# Populate the community assignment array
for node in G:
    # Directly access the community of the node
    node_community = G.nodes[node]['community']
    node_community = sorted(node_community)
    
    # In the case of non-overlapping communities, node_community is an integer
    community_assignment[node] = node_community[0]

# Display results
# print(f"Adjacency Matrix:\n", adj_matrix)
print(f"\nNumber of nodes: {adj_matrix.shape[0]}")
print(f"Number of edges: {np.sum(adj_matrix) // 2}")
print(f"\nNumber of unique communities: {len(np.unique(community_assignment))}")
print(f"\nOriginal Community Assignment:\n", community_assignment)
features_matrix = np.eye(adj_matrix.shape[0])  # Identity matrix of size n x n
unique_values = np.unique(community_assignment)
# num_communities = estimate_number_of_communities(adj_matrix)
# print(f"Estimated number of communities: {num_communities}")
communities,P=CDNMF(adj_matrix,features_matrix,k=len(unique_values), num_iterations=100, beta=0.5)
print(f"Calculated Community Assignment:\n", communities)
orig_modularity = calculate_modularity(adj_matrix, community_assignment)
calc_modularity = calculate_modularity(adj_matrix, communities)
ari = adjusted_rand_score(community_assignment, communities)
nmi = normalized_mutual_info_score(community_assignment, communities)
acc = best_accuracy(community_assignment, communities)
print(f"Original modularity: {orig_modularity}")
print(f"Calculated modularity: {calc_modularity}")
print(f'Adjusted Rand Index: {ari}')
print(f'Normalized Mutual Information: {nmi}')
print(f'Accuracy: {acc}')


Number of nodes: 3000
Number of edges: 2766.0

Number of unique communities: 41

Original Community Assignment:
 [ 0  1  2 ... 26 70 67]
Estimated number of communities: 41
Objective function value after 1 iterations: 115.25257786861737
Objective function value after 11 iterations: 71.75536246517768
Objective function value after 21 iterations: 71.36989229378278
Objective function value after 31 iterations: 71.35428653230416
Objective function value after 41 iterations: 71.32792059150681
Objective function value after 51 iterations: 71.31248225262429
Objective function value after 61 iterations: 71.2914337891375
Converged after 65 iterations.
Calculated Community Assignment:
 [ 4 40  5 ... 31 16  3]
Original modularity: 0.9302917907716919
Calculated modularity: 0.9249883933032292
Adjusted Rand Index: 0.16723139142068377
Normalized Mutual Information: 0.4792574530952167
Accuracy: 0.33866666666666667
